# Análisis de Satisfacción Laboral

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Cargar el dataset
df = pd.read_csv('database.csv')

## Análisis Exploratorio de Datos (EDA)

In [ ]:
# Mostrar primeras filas
print("Primeras 5 filas del dataset:")
print(df.head())

# Información general y tipos de datos
print("\nInformación del dataset:")
df.info()

# Estadísticas descriptivas
print("\nEstadísticas descriptivas:")
print(df.describe())

In [ ]:
# Porcentaje de valores nulos por columna
null_percentage = (df.isnull().sum() / len(df)) * 100
print("\nPorcentaje de valores nulos por columna:")
print(null_percentage.sort_values(ascending=False))

### Visualización de Variables Clave

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='¿Qué tan conforme estás con tus ingresos laborales?', data=df)
plt.title('Distribución de la Satisfacción con los Ingresos Laborales')
plt.xlabel('Nivel de Conformidad')
plt.ylabel('Cantidad')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Limpiando la columna de sueldo para el histograma
df_cleaned = df.dropna(subset=['sueldo_bruto_en_dolares'])
df_cleaned['sueldo_bruto_en_dolares'] = pd.to_numeric(df_cleaned['sueldo_bruto_en_dolares'], errors='coerce')
df_cleaned = df_cleaned.dropna(subset=['sueldo_bruto_en_dolares'])


plt.figure(figsize=(10, 6))
sns.histplot(df_cleaned['sueldo_bruto_en_dolares'], bins=50, kde=True)
plt.title('Histograma de Sueldo Bruto en Dólares')
plt.xlabel('Sueldo Bruto en Dólares')
plt.ylabel('Frecuencia')
plt.xlim(0, 20000) # Ajustar límite para mejor visualización
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(x='seniority', y='sueldo_bruto_en_dolares', data=df_cleaned)
plt.title('Relación entre Seniority y Sueldo Bruto en Dólares')
plt.xlabel('Seniority')
plt.ylabel('Sueldo Bruto en Dólares')
plt.xticks(rotation=45)
plt.ylim(0, 30000) # Ajustar límite para mejor visualización
plt.show()

## Preprocesamiento y Feature Engineering

In [ ]:
# Definición de variables
features = [
    'tengo_edad',
    'ultimo_salario_mensual_o_retiro_bruto_en_pesos_argentinos',
    'Años de experiencia',
    'antiguedad_en_la_empresa_actual',
    'maximo_nivel_de_estudios'
target = '¿Qué tan conforme estás con tus ingresos laborales?'

# Crear un nuevo DataFrame con las columnas de interés
df_model = df[features + [target]].copy()

# --- Limpieza de Datos ---

# Manejar valores faltantes en numéricas con la mediana
for col in ['tengo_edad', 'ultimo_salario_mensual_o_retiro_bruto_en_pesos_argentinos', 'Años de experiencia', 'antiguedad_en_la_empresa_actual']:
    median_val = df_model[col].median()
    df_model[col].fillna(median_val, inplace=True)

# Codificar variables categóricas
label_encoder_studies = LabelEncoder()
df_model['maximo_nivel_de_estudios'] = label_encoder_studies.fit_transform(df_model['maximo_nivel_de_estudios'].astype(str))

# Codificar la variable objetivo
label_encoder_target = LabelEncoder()
df_model[target] = label_encoder_target.fit_transform(df_model[target].astype(str))

# Verificar que no queden nulos
print("Valores nulos después de la limpieza:")
print(df_model.isnull().sum())

# Definir X e y
X = df_model[features]
y = df_model[target]

In [ ]:
# División de Datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape[0]} muestras")
print(f"Tamaño del conjunto de prueba: {X_test.shape[0]} muestras")

## Modelado - Random Forest

In [ ]:
# Inicializar y entrenar el modelo
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

In [ ]:
# Realizar predicciones
y_pred = rf_model.predict(X_test)

### Evaluación del Modelo

In [ ]:
# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder_target.classes_, yticklabels=label_encoder_target.classes_)
plt.title('Matriz de Confusión')
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.show()

In [ ]:
# Calcular métricas
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (Weighted): {precision:.4f}")
print(f"Recall (Weighted): {recall:.4f}")
print(f"F1-Score (Weighted): {f1:.4f}")

## Simulación de Predicción Interactiva

In [ ]:
# Simulación de entrada de un nuevo "usuario"
simulated_input = {
    'tengo_edad': [35],
    'ultimo_salario_mensual_o_retiro_bruto_en_pesos_argentinos': [500000],
    'Años de experiencia': [10],
    'antiguedad_en_la_empresa_actual': [3],
    'maximo_nivel_de_estudios': ['Universitario'] # Usar una categoría original antes de la codificación
}

# Crear DataFrame para la simulación
input_df = pd.DataFrame(simulated_input)

# Aplicar la misma codificación que a los datos de entrenamiento
input_df['maximo_nivel_de_estudios'] = label_encoder_studies.transform(input_df['maximo_nivel_de_estudios'])

# Realizar la predicción de probabilidades
prediction_proba = rf_model.predict_proba(input_df)

# Mostrar los resultados
print("Datos de entrada simulados:")
print(simulated_input)
print("\nProbabilidades de satisfacción laboral:")
for i, class_name in enumerate(label_encoder_target.classes_):
    print(f"- {class_name}: {prediction_proba[0][i]*100:.2f}%")

# Obtener la predicción final
predicted_class_index = np.argmax(prediction_proba)
predicted_class_name = label_encoder_target.classes_[predicted_class_index]
print(f"\nPredicción final: El usuario probablemente está '{predicted_class_name}' con su salario.")